# 02 Data Wrangling and Visualization — Reference Solutions

Complete solutions to the Songbai Nursing Home Legionnaires' disease line-list exercises.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# -- CJK font setup (avoid Chinese labels showing as boxes) --
# Scan the system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# Plotly: make sure interactive charts still render during a static build (jupyter-book build)
pio.renderers.default = "notebook"


## Question 1: Read in and inspect the data

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
print(f"Data dimensions: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumn names: {df.columns.tolist()}")
df.head()

In [ ]:
df.info()

## Question 2: Date conversion and derived variables

In [ ]:
# Date conversion
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")

# Create the infected column
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# Compute onset_to_hosp_days
df["onset_to_hosp_days"] = (
    df["hospitalization_date"] - df["symptom_onset_date"]
).dt.days

# Print the first 10 infected people
infected_df = df[df["infected"] == 1]
infected_df[["case_id", "symptom_onset_date", "onset_to_hosp_days"]].head(10)

## Question 3: Epidemic curve

In [ ]:
import matplotlib.dates as mdates

cases = df[df["infected"] == 1]
daily = cases.groupby("symptom_onset_date").size().rename("cases")

# Add the pre-outbreak background period (including zero-case days)
date_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),
    daily.index.max() + pd.Timedelta(days=1),
)
daily = daily.reindex(date_range, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    daily.index, daily.values,
    width=1.0,
    color="#2c7fb8", edgecolor="white", linewidth=0.5,
)
ax.set_title(
    "Songbai Nursing Home Legionnaires' Disease Epidemic Curve, by Date of Symptom Onset, January 2026",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## Question 4: Attack rate by wing comparison chart

In [ ]:
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(residents=("case_id", "size"), infected=("infected", "sum"))
    .reset_index()
)
wing_stats["attack_rate_pct"] = (
    wing_stats["infected"] / wing_stats["residents"] * 100
).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]
wing_stats = wing_stats.sort_values("attack_rate_pct", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=wing_stats, x="label", y="attack_rate_pct",
    hue="label", palette="YlOrRd", legend=False, ax=ax,
)
ax.set_title("Attack Rate by Wing")
ax.set_xlabel("Wing")
ax.set_ylabel("Attack Rate (%)")

for i, row in enumerate(wing_stats.itertuples()):
    ax.text(i, row.attack_rate_pct + 1, f"{row.attack_rate_pct}%",
            ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## Question 5 (Challenge): Interactive stratified epidemic curve

In [ ]:
daily_floor = (
    cases.groupby(["symptom_onset_date", "floor"])
    .size()
    .rename("cases")
    .reset_index()
)
daily_floor["floor"] = daily_floor["floor"].astype(str) + "F"

fig = px.bar(
    daily_floor,
    x="symptom_onset_date", y="cases", color="floor",
    barmode="stack",
    title="Songbai Nursing Home Legionnaires' Disease Epidemic Curve (Stratified by Floor), January 2026",
    labels={"symptom_onset_date": "Date of Symptom Onset", "cases": "Number of Cases", "floor": "Floor"},
)
fig.update_layout(
    bargap=0,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False, rangemode="tozero"),
    plot_bgcolor="white",
)
fig.show()

### Interpretation

- **Epidemic curve**: cases peak within a few days, showing the classic **point-source (common source)** pattern
- **Wing comparison**: wing 3B has the highest attack rate and wing 1B the lowest → the exposure source may be tied to facilities in a specific area
- **Stratified curve**: if the third floor peaks before the first floor, it may hint that the exposure source is on a higher floor (e.g. the water-tower supply piping)

## Question 6: Frequency table and pivot table

In [ ]:
# Severity frequency distribution
print("=== Severity frequency distribution ===")
print(df["clinical_severity"].value_counts())
print("\n=== Severity percentages ===")
print(df["clinical_severity"].value_counts(normalize=True).mul(100).round(1))

# Wing × floor attack-rate table
print("\n=== Attack rate by wing × floor ===")
pivot = pd.pivot_table(
    df,
    values="infected",
    index="wing",
    columns="floor",
    aggfunc="mean",
    margins=True,
)
print(pivot.round(3))

## Question 7: Method Chaining

In [ ]:
# Use method chaining to do it in one go: filter infected 70+ → group by floor → count cases and deaths → CFR → sort
result = (
    df
    .query("infected == 1 and age >= 70")
    .groupby("floor")
    .agg(
        n_cases=("case_id", "count"),
        n_deaths=("outcome", lambda x: (x == "dead").sum()),
    )
    .assign(cfr=lambda d: (d["n_deaths"] / d["n_cases"] * 100).round(1))
    .sort_values("cfr", ascending=False)
)
print("Case fatality rate by floor, infected people aged 70+:")
result

## Question 8: Joining data and text cleaning

In [ ]:
import numpy as np

# 1. Simulate lab-results data
rng = np.random.default_rng(42)
infected_ids = df.loc[df["infected"] == 1, "case_id"].tolist()
lab = pd.DataFrame({
    "case_id": infected_ids,
    "ct_value": rng.uniform(15, 35, size=len(infected_ids)).round(1),
})
print(f"Lab data: {len(lab)} rows")
lab.head()

In [ ]:
# 2. Merge the lab data
df_merged = pd.merge(df, lab, on="case_id", how="left")
print(f"After merge: {df_merged.shape}")
print(f"Rows with a ct_value: {df_merged['ct_value'].notna().sum()}")

# 3. Standardize wing case
df_merged["wing"] = df_merged["wing"].str.strip().str.upper()
print(f"\nWing categories: {df_merged['wing'].unique()}")

# 4. Remove duplicate reports
before = len(df_merged)
df_merged = df_merged.drop_duplicates(subset="case_id", keep="first")
print(f"Before dedup: {before}, after dedup: {len(df_merged)}")

# 5. The 5 oldest infected people
top5 = df_merged.query("infected == 1").nlargest(5, "age")
print("\nThe 5 oldest infected people:")
top5[["case_id", "age", "floor", "wing", "clinical_severity", "ct_value"]]